# DermaScope - Notebook Pengolahan Citra

Notebook ini dipakai untuk uji awal dan bahan screenshot. Alurnya sama dengan versi web: upload foto, crop wajah, pra-pemrosesan, ekstraksi sinyal kulit, overlay, lalu tabel hasil.

Metodenya berbasis pengolahan citra. Nilai dihitung dari piksel gambar: warna, intensitas, tekstur, mask, connected component, dan zona wajah.

## 1. Instalasi Library

OpenCV dipakai untuk pengolahan citra. NumPy untuk operasi piksel. Pandas untuk tabel. Matplotlib untuk menampilkan gambar. scikit-image dipakai untuk fitur tekstur GLCM.

In [ ]:
!pip -q install opencv-python-headless numpy pandas matplotlib scikit-image

## 2. Import dan Konfigurasi Awal

In [ ]:
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from skimage.feature import graycomatrix, graycoprops

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

CONDITION_META = {
    "acne": {"label": "Jerawat", "short": "Acne", "color": (31, 90, 255), "weight": 2.2, "count_weight": 1.8},
    "dark_spots": {"label": "Noda gelap", "short": "Spot", "color": (0, 77, 138), "weight": 2.0, "count_weight": 0.8},
    "wrinkles": {"label": "Kerutan", "short": "Line", "color": (143, 123, 0), "weight": 1.7, "count_weight": 0.15},
    "redness": {"label": "Kemerahan", "short": "Red", "color": (127, 0, 209), "weight": 1.6, "count_weight": 0.25},
    "pores": {"label": "Pori besar", "short": "Pore", "color": (31, 122, 97), "weight": 1.4, "count_weight": 0.08},
}

def show_bgr(title, image, size=(6, 6)):
    plt.figure(figsize=size)
    plt.imshow(cv.cvtColor(image, cv.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 3. Fungsi Pra-pemrosesan Citra Wajah

Kode ini bisa dipakai sebagai **Gambar 1.1 Kode Pra-pemrosesan Citra Wajah**. Prosesnya: deteksi wajah, crop area wajah, lalu normalisasi cahaya.

In [ ]:
def detect_face(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    cascade_path = cv.data.haarcascades + "haarcascade_frontalface_default.xml"
    cascade = cv.CascadeClassifier(cascade_path)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda item: item[2] * item[3])
        return (int(x), int(y), int(w), int(h)), True

    tinggi, lebar = image.shape[:2]
    ukuran = int(min(lebar, tinggi) * 0.72)
    x = max(0, (lebar - ukuran) // 2)
    y = max(0, int((tinggi - ukuran) * 0.42))
    return (x, y, min(ukuran, lebar - x), min(ukuran, tinggi - y)), False


def expand_face_rect(face, image_width, image_height):
    x, y, w, h = face
    pad_x = int(w * 0.26)
    pad_y_top = int(h * 0.30)
    pad_y_bottom = int(h * 0.18)
    left = max(0, x - pad_x)
    top = max(0, y - pad_y_top)
    right = min(image_width, x + w + pad_x)
    bottom = min(image_height, y + h + pad_y_bottom)
    return left, top, max(1, right - left), max(1, bottom - top)


def normalize_lighting(face_image):
    lab = cv.cvtColor(face_image, cv.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv.split(lab)
    clahe = cv.createCLAHE(clipLimit=1.8, tileGridSize=(8, 8))
    normalized_l = clahe.apply(l_channel)
    normalized_lab = cv.merge((normalized_l, a_channel, b_channel))
    return cv.cvtColor(normalized_lab, cv.COLOR_LAB2BGR)


def preprocess_face(image):
    face, detected = detect_face(image)
    crop_x, crop_y, crop_w, crop_h = expand_face_rect(face, image.shape[1], image.shape[0])
    face_crop = image[crop_y:crop_y + crop_h, crop_x:crop_x + crop_w]
    preprocessed = normalize_lighting(face_crop)
    return preprocessed, face, (crop_x, crop_y, crop_w, crop_h), detected

## 4. Fungsi Ekstraksi Sinyal Visual Kulit

Kode ini bisa dipakai sebagai **Gambar 1.2 Kode Ekstraksi Sinyal Visual Kulit**. Sinyal dihitung dari mask kulit, threshold per zona, morfologi, connected components, dan GLCM.

In [ ]:
def skin_mask(face):
    ycrcb = cv.cvtColor(face, cv.COLOR_BGR2YCrCb)
    hsv = cv.cvtColor(face, cv.COLOR_BGR2HSV)
    color_mask = cv.inRange(ycrcb, np.array([30, 128, 75]), np.array([245, 188, 148]))
    hsv_mask = cv.inRange(hsv, np.array([0, 18, 45]), np.array([35, 185, 255]))
    mask = cv.bitwise_and(color_mask, hsv_mask)
    kernel = np.ones((5, 5), dtype=np.uint8)
    mask = cv.morphologyEx(mask, cv.MORPH_OPEN, kernel)
    mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, kernel)

    if np.count_nonzero(mask) < face.shape[0] * face.shape[1] * 0.12:
        mask = np.zeros(face.shape[:2], dtype=np.uint8)
        center = (face.shape[1] // 2, int(face.shape[0] * 0.54))
        axes = (int(face.shape[1] * 0.36), int(face.shape[0] * 0.42))
        cv.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
    return mask


def zone_rects(width, height):
    return [
        ("forehead", "Dahi", (int(width * 0.20), int(height * 0.08), int(width * 0.60), int(height * 0.20))),
        ("left_cheek", "Pipi kiri", (0, int(height * 0.32), int(width * 0.42), int(height * 0.38))),
        ("right_cheek", "Pipi kanan", (int(width * 0.58), int(height * 0.32), int(width * 0.42), int(height * 0.38))),
        ("nose", "Hidung", (int(width * 0.38), int(height * 0.30), int(width * 0.24), int(height * 0.44))),
        ("chin", "Dagu", (int(width * 0.26), int(height * 0.70), int(width * 0.48), int(height * 0.30))),
    ]


def small_blob_mask(mask, min_area, max_area):
    count, labels, stats, _ = cv.connectedComponentsWithStats(mask, connectivity=8)
    output = np.zeros_like(mask)
    for index in range(1, count):
        area = int(stats[index, cv.CC_STAT_AREA])
        if min_area <= area <= max_area:
            output[labels == index] = 255
    return output


def measurement(mask, skin):
    skin_area = max(1, int(np.count_nonzero(skin)))
    count, labels, stats, _ = cv.connectedComponentsWithStats(mask, connectivity=8)
    areas = [int(stats[index, cv.CC_STAT_AREA]) for index in range(1, count)]
    active = sum(areas)
    return {
        "coverage": round((active / skin_area) * 100, 2),
        "count": len([area for area in areas if area >= 4]),
        "mask": mask,
    }


def glcm_texture(gray, valid_mask):
    sample = cv.resize(gray, (128, 128), interpolation=cv.INTER_AREA)
    sample = (sample / 4).astype(np.uint8)
    glcm = graycomatrix(sample, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)
    return {
        "glcm_contrast": round(float(graycoprops(glcm, "contrast")[0, 0]), 4),
        "glcm_energy": round(float(graycoprops(glcm, "energy")[0, 0]), 4),
        "glcm_homogeneity": round(float(graycoprops(glcm, "homogeneity")[0, 0]), 4),
        "glcm_correlation": round(float(graycoprops(glcm, "correlation")[0, 0]), 4),
    }


def measure_zone(zone, skin):
    b, g, r = cv.split(zone)
    gray = cv.cvtColor(zone, cv.COLOR_BGR2GRAY)
    valid = skin > 0
    baseline = float(np.median(gray[valid])) if np.any(valid) else float(np.median(gray))
    red_dominance = r.astype(np.int16) - ((g.astype(np.int16) + b.astype(np.int16)) // 2)
    red_base = float(np.median(red_dominance[valid])) if np.any(valid) else float(np.median(red_dominance))

    red_strong = red_dominance > max(30, red_base + 24)
    red_soft = red_dominance > max(24, red_base + 18)

    acne = np.where(valid & red_strong & (gray < baseline + 28), 255, 0).astype(np.uint8)
    acne = small_blob_mask(acne, min_area=8, max_area=160)

    dark = np.where(valid & (gray < baseline - 26) & ~red_strong, 255, 0).astype(np.uint8)
    dark = small_blob_mask(dark, min_area=12, max_area=380)

    redness = np.where(valid & red_soft & (gray > baseline - 18), 255, 0).astype(np.uint8)
    redness = cv.morphologyEx(redness, cv.MORPH_OPEN, np.ones((5, 5), dtype=np.uint8))
    redness = cv.morphologyEx(redness, cv.MORPH_CLOSE, np.ones((7, 7), dtype=np.uint8))

    edges = cv.Canny(cv.GaussianBlur(gray, (3, 3), 0), 45, 120)
    line_kernel = cv.getStructuringElement(cv.MORPH_RECT, (5, 1))
    wrinkles = np.where(valid & (edges > 0) & (gray < baseline + 24), 255, 0).astype(np.uint8)
    wrinkles = cv.morphologyEx(wrinkles, cv.MORPH_OPEN, line_kernel)

    laplacian = cv.Laplacian(gray, cv.CV_16S, ksize=3)
    texture = cv.convertScaleAbs(laplacian)
    texture_base = float(np.percentile(texture[valid], 68)) if np.any(valid) else float(np.percentile(texture, 68))
    pores = np.where(valid & (texture > max(28, texture_base + 8)) & (red_dominance < red_base + 32), 255, 0).astype(np.uint8)
    pores = cv.morphologyEx(pores, cv.MORPH_OPEN, np.ones((2, 2), dtype=np.uint8))

    return {
        "acne": measurement(acne, skin),
        "dark_spots": measurement(dark, skin),
        "wrinkles": measurement(wrinkles, skin),
        "redness": measurement(redness, skin),
        "pores": measurement(pores, skin),
        "texture": glcm_texture(gray, valid),
    }

## 5. Fungsi Analisis Lengkap dan Overlay

In [ ]:
def score_from_penalty(penalty):
    return int(max(0, min(100, round(100 - penalty))))


def put_label(image, text, origin, color, scale=0.45):
    x, y = origin
    font = cv.FONT_HERSHEY_SIMPLEX
    (width, height), baseline = cv.getTextSize(text, font, scale, 1)
    x = max(0, min(x, image.shape[1] - width - 6))
    y = max(height + 4, min(y, image.shape[0] - baseline - 4))
    cv.rectangle(image, (x, y - height - 5), (x + width + 6, y + baseline + 3), color, -1)
    cv.putText(image, text, (x + 3, y), font, scale, (255, 255, 255), 1, cv.LINE_AA)


def render_overlay(face_image, condition_masks):
    result = face_image.copy()
    color_layer = np.zeros_like(result)
    for key, mask in condition_masks.items():
        color_layer[mask > 0] = CONDITION_META[key]["color"]

    blended = cv.addWeighted(result, 0.86, color_layer, 0.30, 0)
    result[np.any(color_layer > 0, axis=2)] = blended[np.any(color_layer > 0, axis=2)]

    tinggi, lebar = result.shape[:2]
    survey_color = (180, 166, 0)
    cv.rectangle(result, (2, 2), (lebar - 3, tinggi - 3), survey_color, 2)
    put_label(result, "Face ROI", (10, 24), survey_color)

    for _, label, (zx, zy, zw, zh) in zone_rects(lebar, tinggi):
        cv.rectangle(result, (zx, zy), (zx + zw, zy + zh), survey_color, 1)
        put_label(result, label, (zx + 4, zy + 16), survey_color, scale=0.38)

    for key, mask in condition_masks.items():
        color = CONDITION_META[key]["color"]
        short = CONDITION_META[key]["short"]
        contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
        for index, contour in enumerate(sorted(contours, key=cv.contourArea, reverse=True)[:8]):
            if cv.contourArea(contour) < 5:
                continue
            x, y, w, h = cv.boundingRect(contour)
            cv.rectangle(result, (x, y), (x + w, y + h), color, 2)
            if index < 2:
                put_label(result, short, (x, max(14, y - 4)), color, scale=0.38)
    return result


def analyze_image(file_name, image):
    preprocessed, _face, _crop, detected = preprocess_face(image)
    mask = skin_mask(preprocessed)
    tinggi, lebar = preprocessed.shape[:2]

    accumulator = {key: {"coverage": 0.0, "count": 0} for key in CONDITION_META}
    condition_masks = {key: np.zeros((tinggi, lebar), dtype=np.uint8) for key in CONDITION_META}
    zone_rows = []
    texture_rows = []

    for zone_id, zone_label, (zx, zy, zw, zh) in zone_rects(lebar, tinggi):
        zone_img = preprocessed[zy:zy + zh, zx:zx + zw]
        zone_skin = mask[zy:zy + zh, zx:zx + zw]
        hasil = measure_zone(zone_img, zone_skin)

        concerns = {}
        for key in CONDITION_META:
            coverage = float(hasil[key]["coverage"])
            count = int(hasil[key]["count"])
            accumulator[key]["coverage"] += coverage
            accumulator[key]["count"] += count
            concerns[key] = coverage
            condition_masks[key][zy:zy + zh, zx:zx + zw][hasil[key]["mask"] > 0] = 255

        zone_penalty = min(65.0, sum(concerns.values()) * 1.8)
        dominant_key = max(concerns, key=concerns.get)
        zone_rows.append({
            "nama_berkas": file_name,
            "zona_wajah": zone_label,
            "score_zona": score_from_penalty(zone_penalty),
            "sinyal_dominan": CONDITION_META[dominant_key]["label"],
            "skin_pixels": int(np.count_nonzero(zone_skin)),
        })

        texture_rows.append({"nama_berkas": file_name, "zona_wajah": zone_label, **hasil["texture"]})

    feature_rows = []
    for key, meta in CONDITION_META.items():
        avg_coverage = accumulator[key]["coverage"] / len(zone_rects(lebar, tinggi))
        count = int(accumulator[key]["count"])
        penalty = min(82.0, avg_coverage * meta["weight"] + min(24.0, count * meta["count_weight"]))
        feature_rows.append({
            "nama_berkas": file_name,
            "fitur_sinyal": meta["label"],
            "score": score_from_penalty(penalty),
            "coverage_area": round(avg_coverage, 2),
            "count_titik": count,
        })

    overlay = render_overlay(preprocessed, condition_masks)
    total_score = round(np.mean([row["score"] for row in feature_rows]))
    summary = {"nama_berkas": file_name, "face_detected": detected, "skin_health_score": total_score}
    return preprocessed, overlay, feature_rows, zone_rows, texture_rows, summary

## 6. Upload Foto dan Jalankan Analisis

Upload maksimal 5 foto. Jika lebih dari 5, notebook hanya memproses 5 foto pertama agar tabel tetap rapi untuk laporan.


In [ ]:
uploaded = files.upload()
selected_files = list(uploaded.items())[:5]

if len(uploaded) > 5:
    print("Catatan: hanya 5 foto pertama yang diproses.")

if not selected_files:
    raise ValueError("Upload minimal 1 gambar.")

all_feature_rows = []
all_zone_rows = []
all_texture_rows = []
all_summary_rows = []

def show_sample_result(file_name, image, preprocessed, overlay, summary):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    items = [
        ("Citra Original", image),
        ("Hasil Pra-pemrosesan", preprocessed),
        ("Output Overlay", overlay),
    ]
    for ax, (title, img) in zip(axes, items):
        ax.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    status = "YES" if summary["face_detected"] else "NO"
    fig.suptitle(f"{file_name} | Skin Health Score: {summary['skin_health_score']}/100 | Face Detected: {status}", fontsize=11)
    plt.tight_layout()
    plt.show()

for file_name, file_bytes in selected_files:
    image = cv.imdecode(np.frombuffer(file_bytes, np.uint8), cv.IMREAD_COLOR)
    if image is None:
        print(f"File gagal dibaca: {file_name}")
        continue

    preprocessed, overlay, feature_rows, zone_rows, texture_rows, summary = analyze_image(file_name, image)
    all_feature_rows.extend(feature_rows)
    all_zone_rows.extend(zone_rows)
    all_texture_rows.extend(texture_rows)
    all_summary_rows.append(summary)

    show_sample_result(file_name, image, preprocessed, overlay, summary)

print(f"Jumlah foto diproses: {len(all_summary_rows)}")


## 7. Tabel Data Hasil Analisis

Tabel dibuat dalam format ringkas untuk laporan. Jika 5 foto diunggah, semua data 5 foto akan tampil.


In [ ]:
df_summary = pd.DataFrame(all_summary_rows)
df_features = pd.DataFrame(all_feature_rows)
df_zones = pd.DataFrame(all_zone_rows)
df_texture = pd.DataFrame(all_texture_rows)

if df_summary.empty or df_features.empty:
    raise ValueError("Belum ada data analisis. Jalankan cell upload terlebih dahulu.")

feature_order = ["Jerawat", "Noda gelap", "Kerutan", "Kemerahan", "Pori besar"]

summary_table = df_summary.rename(columns={
    "nama_berkas": "Nama Berkas",
    "face_detected": "Face Detected",
    "skin_health_score": "Skin Health Score",
})
summary_table["Face Detected"] = summary_table["Face Detected"].map({True: "YES", False: "NO"})

score_table = df_features.pivot_table(
    index="nama_berkas", columns="fitur_sinyal", values="score", aggfunc="first"
).reindex(columns=feature_order).reset_index().rename(columns={"nama_berkas": "Nama Berkas"})

coverage_table = df_features.pivot_table(
    index="nama_berkas", columns="fitur_sinyal", values="coverage_area", aggfunc="first"
).reindex(columns=feature_order).round(2).reset_index().rename(columns={"nama_berkas": "Nama Berkas"})

count_table = df_features.pivot_table(
    index="nama_berkas", columns="fitur_sinyal", values="count_titik", aggfunc="first"
).reindex(columns=feature_order).fillna(0).astype({name: int for name in feature_order}).reset_index().rename(columns={"nama_berkas": "Nama Berkas"})

zone_table = df_zones.rename(columns={
    "nama_berkas": "Nama Berkas",
    "zona_wajah": "Zona Wajah",
    "score_zona": "Score Zona",
    "sinyal_dominan": "Sinyal Dominan",
    "skin_pixels": "Skin Pixels",
}).reset_index(drop=True)

texture_table = df_texture.rename(columns={
    "nama_berkas": "Nama Berkas",
    "zona_wajah": "Zona Wajah",
    "glcm_contrast": "Contrast",
    "glcm_energy": "Energy",
    "glcm_homogeneity": "Homogeneity",
    "glcm_correlation": "Correlation",
}).round(4).reset_index(drop=True)

std_table = df_features.groupby("fitur_sinyal")[["score", "coverage_area", "count_titik"]].std(ddof=0).reset_index()
std_table = std_table.rename(columns={
    "fitur_sinyal": "Fitur Sinyal",
    "score": "Std. Score",
    "coverage_area": "Std. Coverage",
    "count_titik": "Std. Count",
}).round(4)

print("A. Ringkasan Hasil per Foto")
display(summary_table)

print("B. Data Fitur Hasil Ekstraksi Sinyal Kulit - Score")
display(score_table)

print("C. Data Coverage Area (%)")
display(coverage_table)

print("D. Data Count / Jumlah Titik")
display(count_table)

print("E. Data Zona Wajah")
display(zone_table)

print("F. Data Tekstur GLCM per Zona")
display(texture_table)

print("G. Hasil Standar Deviasi")
display(std_table)


## Catatan Metode

Metode yang dipakai pada notebook ini:

1. **Akuisisi citra**: gambar wajah diunggah sebagai input.
2. **Deteksi wajah Haar Cascade**: mencari posisi wajah untuk menentukan ROI.
3. **Crop ROI wajah**: memotong area wajah agar background tidak ikut dihitung.
4. **Normalisasi cahaya CLAHE**: meratakan kontras agar foto lebih stabil dibaca.
5. **Konversi ruang warna**: memakai BGR, grayscale, HSV, YCrCb, dan LAB sesuai kebutuhan proses.
6. **Skin mask**: memisahkan area kulit dari area non-kulit.
7. **Pembagian zona wajah**: membagi wajah menjadi dahi, pipi kiri, pipi kanan, hidung, dan dagu.
8. **Threshold adaptif**: membandingkan piksel dengan nilai median zona, bukan angka tetap global.
9. **Morfologi citra**: memakai opening dan closing untuk membersihkan noise.
10. **Connected components**: menghitung jumlah titik atau area sinyal yang terhubung.
11. **Canny edge detection**: membaca garis halus untuk sinyal kerutan.
12. **Laplacian texture response**: membaca detail tekstur kecil untuk sinyal pori.
13. **GLCM**: menghitung fitur tekstur seperti contrast, energy, homogeneity, dan correlation.
14. **Overlay visual**: memberi tanda warna pada sinyal yang terbaca di wajah.
15. **Perhitungan coverage, count, score, dan standar deviasi**: mengubah hasil mask menjadi data numerik.

Batasannya: hasil ini adalah pemetaan sinyal visual pada citra, bukan diagnosis medis.
